In [ ]:
#LOOKFORWARD YEARLY LOGISTIC ANALYSIS - PER TEST COGNITION 

"""
DYNAMIC look-forward landmark logistic, per test (all data, most-recent origin).
Same logic as the composite cognition - but applied per test
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import seaborn as sns

#minimal plotting size
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['axes.edgecolor'] = 'black'
plt.rcParams['axes.linewidth'] = 0.8

output_dir = 'lookforward_dynamic_pertest_int'
os.makedirs(output_dir, exist_ok=True)

data = pd.read_csv('cv_results_all_tests_merged_with_groups.csv')
data = data[data['test_name'] != 'PMem'].copy()

n_prev = (data['prevalent'] == 1).sum()
data = data[data['prevalent'] == 0].copy()
print(f"Dropped {n_prev:,} prevalent observations")

test_direction = {
    'PMat': -1, 'RT': -1, 'FI': 1, 'NM': 1, 'SD': 1, 'TMA': -1,
    'TMB': -1, 'MP': 1, 'PAL': 1, 'PV': 1, 'TR': 1, 'BL': 1
}

data['z_score_adj'] = -(data['z_score'] * data['test_name'].map(test_direction))
data = data[data['time_to_event'] > 0].dropna(
    subset=['z_score', 'age_at_test', 'time_to_event', 'event']).copy()

adm = (data.loc[data.event == 0, 'birth_decimal']
       + data.loc[data.event == 0, 'age_at_followup_end']).max()
data['potential_fu'] = (adm - data['birth_decimal']) - data['age_at_test']
print(f"Admin censoring date: {adm:.2f}; "
      f"median potential follow-up: {data['potential_fu'].median():.1f} y")

tests = [t for t in data['test_name'].unique()]
forward_windows = list(range(1, 21))
print(f"\nN = {data['eid'].nunique():,}  participants")
print(f"Tests: {tests}")

In [ ]:
#Per-test dynamic look-forward loop in R
"""
-i data -i tests -i forward_windows -o all_results -o risk_curves
For each test x horizon: classify visits, take each person's most recent
classifiable visit, fit logistic, read absolute risk at z = 0, 1, 2.
"""
%load_ext rpy2.ipython

In [ ]:
%%R -i data -i tests -i forward_windows -o all_results -o risk_curves

library(broom)
library(dplyr)

all_results <- list()
risk_curves <- list()
row_idx <- 1

for (test in tests) {
  cat("\nTest:", test, "\n")
  test_df <- data %>%
    filter(test_name == test) %>%
    filter(!is.na(z_score), !is.na(age_at_test),
           !is.na(time_to_event), !is.na(event), time_to_event > 0)

  for (t in forward_windows) {
    classifiable <- test_df %>%
      filter(potential_fu >= t) %>%
      group_by(eid) %>%
      slice_max(age_at_test, n = 1, with_ties = FALSE) %>%
      ungroup() %>%
      mutate(dem_within_t = case_when(
        event == 1 & time_to_event > (t - 1) & time_to_event <= t ~ 1,
        time_to_event >= t             ~ 0,
        TRUE                           ~ NA_real_
      )) %>%
      filter(!is.na(dem_within_t))

    n_cases <- sum(classifiable$dem_within_t)
    if (n_cases < 10 || nrow(classifiable) < 100) next

    fit <- tryCatch(
      glm(dem_within_t ~ z_score_adj + age_at_test, data = classifiable,
          family = binomial(link = "logit")),
      error = function(e) return(NULL))
    if (is.null(fit)) next

    tidy_fit <- tidy(fit, exponentiate = TRUE)        # no conf.int = no profiling
    z_row <- tidy_fit[tidy_fit$term == "z_score_adj", ]
    z_or      <- z_row$estimate
    z_ci_low  <- exp(log(z_or) - 1.96 * z_row$std.error)
    z_ci_high <- exp(log(z_or) + 1.96 * z_row$std.error)

    mean_age <- mean(classifiable$age_at_test)
    newd <- data.frame(z_score_adj = c(0, 1, 2), age_at_test = mean_age)
    newd$predicted_risk <- predict(fit, newdata = newd, type = "response")
    newd$z_label <- c("average", "1 SD worse", "2 SD worse")
    newd$test <- test
    newd$forward_window <- t
    risk_curves[[paste0(test, "_", t)]] <- newd

    all_results[[row_idx]] <- data.frame(
      test = test, forward_window = t, n_total = nrow(classifiable),
      n_cases = n_cases, incidence_pct = 100 * n_cases / nrow(classifiable),
      z_or = z_or, z_ci_low = z_ci_low,
      z_ci_high = z_ci_high, z_pval = z_row$p.value)
    row_idx <- row_idx + 1
  }
  cat("  Done\n")
}

all_results <- do.call(rbind, all_results)
risk_curves <- do.call(rbind, risk_curves)
cat("\nTotal models fit:", nrow(all_results), "\n")

In [ ]:
#Per-test absolute-risk small multiples
all_results = all_results.sort_values(['test', 'forward_window']).reset_index(drop=True)
all_results.to_csv(f'{output_dir}/lookforward_results_dynamic_pertest.csv', index=False)
risk_curves.to_csv(f'{output_dir}/risk_curves_dynamic_pertest.csv', index=False)

risk_curves['predicted_risk_pct'] = risk_curves['predicted_risk'] * 100
tests_fit = sorted(all_results['test'].unique())
n = len(tests_fit); ncol = 4; nrow = int(np.ceil(n / ncol))
colors = {'average': 'black', '1 SD worse': 'darkorange', '2 SD worse': 'firebrick'}

fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 3 * nrow), squeeze=False)
for i, test in enumerate(tests_fit):
    ax = axes[i // ncol][i % ncol]
    g = risk_curves[risk_curves['test'] == test]
    for lab in ['average', '1 SD worse', '2 SD worse']:
        gl = g[g['z_label'] == lab]
        ax.plot(gl['forward_window'], gl['predicted_risk_pct'],
                marker='o', markersize=3, linewidth=1.5, color=colors[lab], label=lab)
    ax.set_title(test, fontsize=11)
    ax.set_xlabel('Years after test', fontsize=9)
    ax.set_ylabel('Risk (%)', fontsize=9)
    ax.grid(alpha=0.3)
    if i == 0:
        ax.legend(fontsize=7, frameon=False)
for j in range(n, nrow * ncol):
    axes[j // ncol][j % ncol].axis('off')
fig.suptitle('Per-test dynamic look-forward absolute risk (all data)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f'{output_dir}/lookforward_dynamic_pertest_grid.png', dpi=800, bbox_inches='tight')
plt.show()